# AI-Powered Incident Diagnosis & Root Cause Analysis Agent 🚨

A Python + GenAI starter that analyzes application logs and service metrics, detects anomalous signals, builds an incident timeline, ranks root-cause hypotheses, and exports an incident report.

**Safety:** This notebook uses synthetic data and is read-only. It does not connect to production, execute remediation, or change infrastructure. Findings are hypotheses requiring human verification.

## Features
- Synthetic logs, metrics, and service dependency data
- Log severity normalization and metric anomaly detection
- Incident timeline and signal correlation
- Evidence-based root-cause hypothesis ranking
- Optional OpenAI-compatible LLM incident summary
- Human-reviewed recommendations and JSON/Markdown exports

In [ ]:
%pip install -q pandas numpy matplotlib python-dotenv openai

In [ ]:
import os, json
from pathlib import Path
from datetime import datetime, timedelta, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

API_KEY = os.getenv("OPENAI_API_KEY", "")
BASE_URL = os.getenv("OPENAI_BASE_URL", "").strip() or None
MODEL = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")
print("LLM configured:", bool(API_KEY), "Model:", MODEL)

## 1. Create sample observability data\nThe demo simulates a checkout API incident with database connection pool pressure.

In [ ]:
rng = np.random.default_rng(17)
start = pd.Timestamp("2026-10-20 10:00:00", tz="UTC")
times = pd.date_range(start, periods=25, freq="2min")
metrics = pd.DataFrame({
    "timestamp": times, "service": "checkout-api",
    "error_rate_pct": np.r_[rng.uniform(.2,.8,8),rng.uniform(18,32,17)],
    "p95_latency_ms": np.r_[rng.normal(180,15,8),rng.normal(1850,180,17)].clip(80),
    "cpu_pct": np.r_[rng.normal(35,4,8),rng.normal(48,6,17)].clip(1,100),
    "db_connections_used": np.r_[rng.integers(20,35,8),rng.integers(92,101,17)],
    "db_connections_limit": 100
})
events = [
 (0,"INFO","checkout-api","Request processing normal"),
 (3,"INFO","deployment","checkout-api version 2.8.1 deployed"),
 (8,"WARN","checkout-api","Database connection acquisition taking longer"),
 (10,"ERROR","checkout-api","Timeout while acquiring database connection"),
 (12,"ERROR","checkout-api","HTTP 503 returned for checkout request"),
 (14,"WARN","database","Connection pool utilization above 95 percent"),
 (16,"ERROR","checkout-api","Database connection pool exhausted"),
 (18,"ERROR","checkout-api","Checkout failed after database timeout"),
 (22,"INFO","platform","Incident investigation started")
]
logs = pd.DataFrame([{"timestamp":start+pd.Timedelta(minutes=m),"level":lv,
 "service":svc,"message":msg} for m,lv,svc,msg in events]).sort_values("timestamp")
dependencies = pd.DataFrame([
 {"upstream":"checkout-api","downstream":"orders-db","relationship":"reads/writes"},
 {"upstream":"checkout-api","downstream":"payment-gateway","relationship":"payment authorization"},
 {"upstream":"web-frontend","downstream":"checkout-api","relationship":"HTTP requests"}
])
display(metrics.head(), logs, dependencies)

## 2. Normalize logs and detect metric anomalies

In [ ]:
severity={"DEBUG":10,"INFO":20,"WARN":30,"ERROR":40,"CRITICAL":50}
logs["severity_score"]=logs["level"].map(severity).fillna(0).astype(int)
logs["is_error"]=logs["severity_score"]>=40

baseline=metrics.iloc[:8]
err_threshold=max(5,baseline.error_rate_pct.mean()+3*baseline.error_rate_pct.std())
lat_threshold=max(500,baseline.p95_latency_ms.mean()+3*baseline.p95_latency_ms.std())
metrics["error_anomaly"]=metrics.error_rate_pct>err_threshold
metrics["latency_anomaly"]=metrics.p95_latency_ms>lat_threshold
metrics["db_pool_pressure"]=metrics.db_connections_used/metrics.db_connections_limit>=.90
metrics["anomaly"]=metrics[["error_anomaly","latency_anomaly","db_pool_pressure"]].any(axis=1)
display(logs[logs.is_error],metrics[metrics.anomaly].head())

## 3. Visualize incident signals

In [ ]:
fig,ax=plt.subplots(3,1,figsize=(12,8),sharex=True)
ax[0].plot(metrics.timestamp,metrics.error_rate_pct,marker="o"); ax[0].set_ylabel("Error %")
ax[1].plot(metrics.timestamp,metrics.p95_latency_ms,marker="o"); ax[1].set_ylabel("P95 ms")
ax[2].plot(metrics.timestamp,metrics.db_connections_used,marker="o")
ax[2].axhline(100,linestyle="--"); ax[2].set_ylabel("DB connections")
plt.tight_layout(); plt.show()

## 4. Build an incident timeline

In [ ]:
log_timeline=logs[["timestamp","service","level","message"]].rename(columns={"level":"signal"})
metric_timeline=metrics[metrics.anomaly][["timestamp","service"]].assign(
    signal="METRIC_ANOMALY",message="One or more metrics exceeded baseline thresholds")
timeline=pd.concat([log_timeline,metric_timeline],ignore_index=True).sort_values("timestamp")
display(timeline)

## 5. Rank root-cause hypotheses\nScores are transparent heuristic rankings, not proof of causality.

In [ ]:
log_text=" ".join(logs.message).lower()
recent=metrics.tail(5)
pool_ratio=(recent.db_connections_used/recent.db_connections_limit).mean()
avg_error=recent.error_rate_pct.mean()
avg_latency=recent.p95_latency_ms.mean()

hypotheses=[]
def add(name,score,evidence,checks,actions):
    hypotheses.append({"hypothesis":name,"score":score,"evidence":evidence,
                       "verification_checks":checks,"suggested_actions":actions})

score=0; evidence=[]
if "connection pool" in log_text or "database connection" in log_text:
    score+=3; evidence.append("Logs mention database connection acquisition/pool exhaustion.")
if pool_ratio>=.90:
    score+=3; evidence.append(f"Recent DB connection utilization averages {pool_ratio:.0%}.")
if avg_error>=10 and avg_latency>=800:
    score+=1; evidence.append("High error rate coincides with elevated latency.")
add("Database connection pool exhaustion",score,evidence,
 ["Inspect active/idle/waiting pool counts and acquisition timeouts.",
  "Check connection leaks and long-running transactions.",
  "Compare pool size with database connection limits."],
 ["Review connection lifecycle and ensure connections are returned.",
  "Inspect slow queries and transaction duration.",
  "Consider pool changes only after validating database capacity."])

deploy="deployed" in log_text
add("Recent deployment regression",2 if deploy else 0,
 ["A deployment event appears in the incident logs."] if deploy else [],
 ["Compare metrics before and after deployment.","Review release and configuration changes."],
 ["Follow the approved rollback process only if the deployment is confirmed as the trigger."])
cpu=float(recent.cpu_pct.mean())
add("CPU saturation",2 if cpu>85 else 0,[f"Recent CPU average is {cpu:.1f}%."] ,
 ["Inspect CPU throttling and process-level CPU."],
 ["Profile hot paths and validate capacity before scaling."])
hypotheses.sort(key=lambda x:x["score"],reverse=True)
for h in hypotheses: print(h["hypothesis"],"score:",h["score"],"evidence:",h["evidence"])

## 6. Optional LLM incident analyst

In [ ]:
SYSTEM_PROMPT = '''
You are an SRE incident-analysis assistant. Analyze only supplied evidence.
Do not invent telemetry or claim a cause is proven. Separate observed facts,
hypotheses, unknowns, and diagnostic checks. Recommend safe read-only checks first.
Return JSON with: incident_summary, observed_facts, ranked_hypotheses,
unknowns, recommended_checks, confidence.
'''

evidence_bundle={
 "service":"checkout-api",
 "error_logs":logs[logs.is_error][["timestamp","service","message"]].astype(str).to_dict("records"),
 "recent_metrics":recent[["error_rate_pct","p95_latency_ms","cpu_pct",
                          "db_connections_used","db_connections_limit"]].mean().to_dict(),
 "rule_hypotheses":hypotheses
}
llm_report=None
if API_KEY:
    try:
        from openai import OpenAI
        kwargs={"api_key":API_KEY}
        if BASE_URL: kwargs["base_url"]=BASE_URL
        client=OpenAI(**kwargs)
        response=client.chat.completions.create(
            model=MODEL,temperature=.2,response_format={"type":"json_object"},
            messages=[{"role":"system","content":SYSTEM_PROMPT},
                      {"role":"user","content":json.dumps(evidence_bundle,default=str)}])
        llm_report=json.loads(response.choices[0].message.content or "{}")
        display(llm_report)
    except Exception as exc:
        print("LLM call unavailable:",exc)
else:
    print("No API key configured; using rule-based analysis only.")

## 7. Create and export the incident report

In [ ]:
incident_report={
 "incident_id":"INC-DEMO-001","service":"checkout-api",
 "status":"investigation_required","generated_at_utc":datetime.now(timezone.utc).isoformat(),
 "summary":"Checkout errors and latency increased alongside database connection pool pressure.",
 "observed_signals":{
  "recent_error_rate_pct_avg":round(float(avg_error),2),
  "recent_p95_latency_ms_avg":round(float(avg_latency),2),
  "recent_db_pool_utilization_pct":round(float(pool_ratio*100),2)},
 "ranked_hypotheses":hypotheses,"llm_analysis":llm_report,
 "human_review_required":True,"automation_performed":False
}
out=Path("incident_output"); out.mkdir(exist_ok=True)
(out/"incident_report.json").write_text(json.dumps(incident_report,indent=2,default=str),encoding="utf-8")
lines=["# Incident Diagnosis Report","",f"Incident: {incident_report['incident_id']}","",
       "## Summary",incident_report["summary"],"","## Ranked Hypotheses"]
for i,h in enumerate(hypotheses,1):
    lines += ["",f"### {i}. {h['hypothesis']} (score {h['score']})"]
    lines += [f"- Evidence: {x}" for x in h["evidence"]]
    lines += [f"- Verify: {x}" for x in h["verification_checks"]]
    lines += [f"- Suggested action: {x}" for x in h["suggested_actions"]]
lines += ["","## Human Review","Validate hypotheses against production telemetry. No remediation was executed."]
(out/"incident_report.md").write_text("\n".join(lines),encoding="utf-8")
print("Exported:",(out/"incident_report.json").resolve())
print("Exported:",(out/"incident_report.md").resolve())

## 8. Production-readiness roadmap

- Ingest approved OpenTelemetry logs, metrics, and traces.
- Correlate distributed traces using trace and span IDs.
- Add RAG over runbooks, service documentation, and historical postmortems.
- Build service dependency graphs and temporal correlation.
- Evaluate hypotheses using labeled incident history.
- Add RBAC, audit logs, redaction, secrets management, and retention controls.
- Integrate ticketing/alerting tools with least-privilege permissions.
- Keep remediation human-approved with dry-run and rollback procedures.

**Evaluation metrics:** Top-1/Top-3 root-cause accuracy, mean time to triage, false-positive rate, evidence coverage, and recommendation safety.